# Tutorial 3 — Operators and Gradients

Fuzzy operators differ in **gradient behaviour**.  The same formula evaluated under different
`Logic` configurations can yield very different learning dynamics.

Swapping operators requires only pass a different `logic=` argument to `Interpretation`.

**Second half of this notebook:** an interactive *connective-surface explorer*. Where the
sections above measure gradient behaviour numerically (loss/`|grad|` on a trained predicate),
the explorer lets you see the same phenomena as a 2-D surface over $[0,1]^2$ — vanishing
gradients as flat plateaus, exploding gradients as sharp cliffs — with sliders for every
operator's hyperparameters ($p$, $\varepsilon$, $\beta$).

> **Extra dependencies for the explorer:** `matplotlib`, `ipywidgets` (`pip install sltn[notebooks]`).

In [ ]:
import torch
import sltn
from sltn import Signature, Real, Interpretation, Logic
import sltn.interpretation.operators.conjunction as conj
import sltn.interpretation.operators.aggregator  as agg

torch.manual_seed(0)

## Setup

Same signature as Tutorial 2.  We train a predicate `P` to test how different operators
affect the loss signal.

In [ ]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Declare the signature
sig = Signature("t2b")
sig.sort("S")
sig.variable("x", "S")
sig.predicate("P", ["S"])

# declare the interpretation of the signature
interp = Interpretation(sig).to(device)

# 2-D real vectors
interp["S"] = sltn.Type("R2", shape=(2,))

data = torch.randn(16, 2, device=device)
interp["x"] = data

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 8), nn.ELU(), nn.Linear(8, 1), nn.Sigmoid())
    def forward(self, v):
        return self.net(v)
interp["P"] = Net()

phi  = sig.parse("forall x: P(x)")   # formula is fixed across all experiments
phi

## 1 — Swapping operators via `Interpretation`

The formula `phi` is unchanged.  A different `logic=` argument selects different fuzzy semantics.

In [ ]:
configs = {
    "Product  + pMeanError (default)": {},
    "Product  + pMean":         {"∀": agg.AggregPMean()},
    "GodelMin + pMeanError":    {"∧": conj.AndMin()},
    "Luk      + pMeanError":    {"∧": conj.AndLuk()},
}

for name, ops in configs.items():
    interp.logic = Logic()
    for symbol, op in ops.items():
        interp.logic[symbol] = op
    out    = interp(phi)          # x resolved from interp["x"] — no assignment dict
    loss   = 1.0 - out.value.mean()
    loss.backward()
    grad_norm = sum(p.grad.norm().item() for p in interp.parameters() if p.grad is not None)
    interp.zero_grad()

    print(f"{name:<40s}  loss={loss.item():.4f}  |grad|={grad_norm:.4f}")

## 2 — Effect of hyperparameter `p` in pMeanError

Higher `p` makes the universal quantifier stricter (approaches min-satisfaction).
`p=1` is the arithmetic mean; `p→∞` is the min.
For learning, `p=2` (default) is a good balance.

In [ ]:
for p in [1, 2, 4, 10]:
    interp.logic["∀"] = agg.AggregPMeanError(p=p)
    r      = interp(phi)
    print(f"p={p:2d}  ∀x P(x) = {r.value.item():.4f}")

## 3 — Visualising connectives as surfaces

The numeric experiments above show *that* gradients differ; this section shows *why*, by
plotting every connective as a 2-D surface over the unit square $[0,1]^2$.

* **Vanishing gradients** — near the boundary $\{0,1\}^2$ many connectives saturate to 0 or 1.
  If your learning signal lives in that region, the gradient of the loss ∝ $1 - \text{sat}$ is
  essentially zero and training stalls.
* **Exploding gradients** — at the other extreme, some connectives have steep slopes near $0$ or
  $1$ that produce very large gradients for small changes in the input.

The `stable` flag (enabled throughout this section) and the `eps` parameter guard against both
problems by clamping inputs away from the boundaries before applying the connective.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sltn.interpretation.operators.utils import pi_0, pi_1

import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Dropdown

### Binary helper operators (for plotting only)

`AggregPMean` and `AggregPMeanError` in sLTN are *aggregation* operators: they reduce an
arbitrary-length sequence (e.g. a whole batch under a quantifier) to a scalar, so they can't be
plotted as a 2-D surface directly. The three classes below are **not part of the sLTN
library** — they are small binary (two-argument) stand-ins defined here purely so the explorer
can plot them as ordinary functions $f : [0,1]^2 \to [0,1]$, side by side with the real
connectives imported from `sltn.interpretation.operators`.

| Class | Formula | Notes |
|-------|---------|-------|
| `BinaryPMean` | $\left(\tfrac{x^p + y^p}{2}\right)^{1/p}$ | Generalised mean; $p=1$ → arithmetic mean, $p \to \infty$ → max |
| `BinaryPMeanError` | $1 - \left(\tfrac{(1-x)^p + (1-y)^p}{2}\right)^{1/p}$ | Dual of pMean; $p=2$ is the default `SatAgg` inside sLTN |
| `BinaryAggProd` | $\exp\!\left(\tfrac{\ln x + \ln y}{2}\right) = \sqrt{xy}$ | Geometric mean; equivalent to product conjunction |

They subclass `ConnectiveOperator` only so they share the explorer's call signature — they are
not registered anywhere in a real `Logic` bundle.

In [ ]:
from sltn.interpretation.operators import ConnectiveOperator

class BinaryPMean(ConnectiveOperator):
    """Generalised (power) mean: ((x^p + y^p) / 2)^(1/p)."""
    def __init__(self, p=1, stable=False, eps=1e-6):
        self.p, self.stable, self.eps = p, stable, eps

    def __repr__(self):
        return f"BinaryPMean(p={self.p}, stable={self.stable})"

    def __call__(self, x, y, p=None, stable=None):
        p = self.p if p is None else p
        stable = self.stable if stable is None else stable
        if stable:
            x, y = pi_0(x, eps=self.eps), pi_0(y, eps=self.eps)
        mean_val = 0.5 * (torch.pow(x, p) + torch.pow(y, p))
        if stable:
            mean_val = pi_0(mean_val, eps=self.eps)
        return torch.pow(mean_val, 1 / p)


class BinaryPMeanError(ConnectiveOperator):
    """Dual pMean error: 1 - pMean(1-x, 1-y). Default SatAgg in sLTN uses p=2."""
    def __init__(self, p=2, stable=False, eps=1e-6):
        self.p, self.stable, self.eps = p, stable, eps

    def __repr__(self):
        return f"BinaryPMeanError(p={self.p}, stable={self.stable})"

    def __call__(self, x, y, p=None, stable=None):
        p = self.p if p is None else p
        stable = self.stable if stable is None else stable
        if stable:
            x, y = pi_1(x, eps=self.eps), pi_1(y, eps=self.eps)
        mean_val = 0.5 * (torch.pow(1 - x, p) + torch.pow(1 - y, p))
        if stable:
            mean_val = pi_0(mean_val)
        return 1 - torch.pow(mean_val, 1 / p)


class BinaryAggProd(ConnectiveOperator):
    """Geometric mean: exp(0.5*(log x + log y)) = sqrt(xy)."""
    def __init__(self, stable=False, eps=1e-6):
        self.stable, self.eps = stable, eps

    def __repr__(self):
        return f"BinaryAggProd(stable={self.stable})"

    def __call__(self, x, y, **_):
        if self.stable:
            x, y = pi_1(x, eps=self.eps), pi_1(y, eps=self.eps)
        return torch.exp(0.5 * (torch.log(x) + torch.log(y)))

### Interactive connective explorer

The cell below renders a contour plot of the selected connective over the grid $[0,1]^2$. Darker
regions are close to 0 (false), lighter regions close to 1 (true).

**What to look for:**

* **Flat plateaus** at 0 or 1 → gradient ≈ 0 → training signal vanishes there.
* **Sharp cliffs** near the axes → gradient spikes → risk of exploding gradients.
* **Effect of *p***: for pMean-based operators, increasing $p$ sharpens the surface towards the max/min behaviour; decreasing towards 0 softens it.
* **Effect of *eps***: the `stable` flag (always on here) clamps inputs by $\varepsilon$, visibly rounding corners near 0 and 1.

Try comparing `And (Prod)` vs `And (Min)` — the product conjunction has a smooth gradient everywhere, while min produces a sharp ridge along the diagonal (this is the `GodelMin` operator compared against `Product` in section 1 above).

In [ ]:
stable = True

def plot_fuzzy_connective(connective_name, p_value, eps_value, beta_value):
    """Render a contour surface for the selected fuzzy connective."""
    operators = {
        "And (Prod)": sltn.interpretation.operators.AndProd(stable=stable, eps=eps_value),
        "And (Min)": sltn.interpretation.operators.AndMin(),
        "And (Luk)": sltn.interpretation.operators.AndLuk(stable=stable, beta=beta_value, smoothing_method="squareplus"),
        "And (pMean)": sltn.interpretation.operators.AndPMean(stable=stable, eps=eps_value, p=p_value),
        "Or (Max)": sltn.interpretation.operators.OrMax(),
        "Or (ProbSum)": sltn.interpretation.operators.OrProbSum(stable=stable, eps=eps_value),
        "Implies (Godel, hard)": sltn.interpretation.operators.ImpliesGodel(smooth=False),
        "Implies (Godel, smooth)": sltn.interpretation.operators.ImpliesGodel(smooth=True, beta=beta_value),
        "Implies (Kleene Dienes)": sltn.interpretation.operators.ImpliesKleeneDienes(),
        "Implies (Reichenbach)": sltn.interpretation.operators.ImpliesReichenbach(stable=stable, eps=eps_value),
        "Implies (Goguen, hard)": sltn.interpretation.operators.ImpliesGoguen(stable=stable, eps=eps_value, smooth=False),
        "Implies (Goguen, smooth)": sltn.interpretation.operators.ImpliesGoguen(stable=stable, eps=eps_value, smooth=True, beta=beta_value),
        "Equiv (Luk)": sltn.interpretation.operators.Equiv(
            and_op=sltn.interpretation.operators.AndProd(stable=stable, eps=eps_value),
            implies_op=sltn.interpretation.operators.ImpliesLuk()),
        "Equiv (Reichenbach)": sltn.interpretation.operators.Equiv(
            and_op=sltn.interpretation.operators.AndProd(stable=stable, eps=eps_value),
            implies_op=sltn.interpretation.operators.ImpliesReichenbach(stable=stable, eps=eps_value)),
        "Equiv (Similarity)": sltn.interpretation.operators.EquivSimilarity(p=p_value, stable=stable, eps=eps_value),
        "p-Mean": BinaryPMean(p=p_value, stable=stable, eps=eps_value),
        "p-Mean Error": BinaryPMeanError(p=p_value, stable=stable, eps=eps_value),
        "AggProd": BinaryAggProd(stable=stable, eps=eps_value),
    }
    op = operators[connective_name]

    X_t = torch.linspace(0, 1, 100)
    Y_t = torch.linspace(0, 1, 100)
    Z_t = op(X_t[None, :], Y_t[:, None]).squeeze()

    fig, ax = plt.subplots(figsize=(6, 5))
    contour = ax.contourf(
        X_t.numpy(), Y_t.numpy(), Z_t.detach().numpy(),
        levels=np.linspace(0, 1, 101), cmap="viridis"
    )
    ax.contour(
        X_t.numpy(), Y_t.numpy(), Z_t.detach().numpy(),
        levels=np.linspace(0, 1, 21), colors="k", linewidths=0.5
    )
    fig.colorbar(contour, ax=ax, ticks=np.linspace(0, 1, 6))
    title = connective_name
    if "p-Mean" in connective_name or "pMean" in connective_name:
        title += f" (p={p_value:.3f}, ε={eps_value:.1e})"
    if "Godel" in connective_name or "Goguen" in connective_name:
        title += f" (beta={beta_value:.1f})"
    ax.set_title(title)
    ax.set_xlabel("Input 1 (x)")
    ax.set_ylabel("Input 2 (y)")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()


interact(
    plot_fuzzy_connective,
    connective_name=Dropdown(
        options=[
            "And (Prod)", "And (Min)", "And (Luk)", "And (pMean)",
            "Or (Max)", "Or (ProbSum)",
            "Implies (Godel, hard)", "Implies (Godel, smooth)",
            "Implies (Kleene Dienes)", "Implies (Reichenbach)", "Implies (Goguen, hard)", "Implies (Goguen, smooth)",
            "Equiv (Luk)", "Equiv (Reichenbach)", "Equiv (Similarity)",
            "p-Mean", "p-Mean Error", "AggProd",
        ],
        value="Implies (Godel, smooth)",
        description="Connective:",
    ),
    p_value=FloatSlider(
        min=0.001, max=10.0, step=0.001, value=2.0,
        description="p:",
        continuous_update=False,
    ),
    eps_value=FloatSlider(
        min=0.0, max=1., step=1e-6, value=1e-6,
        description="eps:",
        continuous_update=False,
    ),
    beta_value=FloatSlider(
        min=0.1, max=10.0, step=0.1, value=10.0,
        description="beta:",
        continuous_update=False,
    ),
)
plt.show()

The `stable` flag is the simplest way to tame gradient issues: it applies the `pi_0` / `pi_1`
clamping functions (see `sltn.interpretation.operators.utils`) to keep inputs away from the
extremes where gradients blow up or vanish.

## 4 — Smooth clamp explorer

Several operators above (`AndLuk`, `ImpliesGodel(smooth=True)`, `ImpliesGoguen(smooth=True)`, …)
use `smooth_clamp(x, beta)` internally instead of a hard `torch.clamp(x, 0, 1)`, precisely to keep
the connective differentiable at the boundary. This section visualises `smooth_clamp` itself and
compares it to the hard clamp.

Use the slider to change `beta` and watch the smooth curve approach the hard clamp as `beta`
increases — larger `beta` means a closer (but less smooth) approximation.

In [ ]:
from sltn.interpretation.operators.utils import smooth_clamp

# Domain for visualization
x_clamp = torch.linspace(-1.5, 2.5, 800)

def plot_smooth_clamp(beta=10.0):
    y_smooth = smooth_clamp(x_clamp, min=0., max=1.0, beta=beta, smoothing_method="softplus")
    y_hard = torch.clamp(x_clamp, 0.0, 1.0)

    plt.figure(figsize=(8, 4.5))
    plt.plot(x_clamp.numpy(), y_smooth.detach().numpy(), label=f"smooth_clamp (beta={beta:.1f})", linewidth=2)
    plt.plot(x_clamp.numpy(), y_hard.numpy(), "--", label="hard clamp [0,1]", linewidth=2)
    plt.ylim(-0.05, 1.05)
    plt.xlim(-1.5, 2.5)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("Smooth Clamp vs Hard Clamp")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()


interact(
    plot_smooth_clamp,
    beta=FloatSlider(value=10.0, min=0., max=20.0, step=0.1, description="beta"),
)
plt.show()

---

**Next:** [Tutorial 4 — KB & Learning](4-KB&Learning.ipynb) puts everything together in an end-to-end training loop over a knowledge base of formulas.